![Henry Logo](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)

# M3L2 E08 - FAISS: guardar vectores y buscar por similitud (Resolution)

## BLOQUE 1 — ¿Qué es FAISS y para qué sirve?

FAISS (**Facebook AI Similarity Search**) es una librería de Meta para búsqueda eficiente de vectores similares.

### El problema que resuelve

En E07 aprendiste que los embeddings son vectores de 1536 dimensiones. Si tenés 10 documentos, podés compararlos a mano con un loop. Pero si tenés **10.000 o 1.000.000** de documentos, comparar cada uno contra la pregunta es inviable.

FAISS construye un **índice** que permite buscar los K vecinos más cercanos en milisegundos, incluso con millones de vectores.

### ¿Qué hace FAISS en el pipeline RAG?

```text
FASE DE INDEXACION:
  Textos -> Embeddings -> FAISS.from_texts() -> Indice FAISS (guardado en disco o memoria)

FASE DE CONSULTA:
  Pregunta -> Embedding -> similarity_search(query, k=2) -> Los 2 textos mas relevantes
```

### Vector stores en LangChain

| Vector Store | Descripción | Cuándo usarlo |
|---|---|---|
| **FAISS** | Índice en memoria de Facebook. Rápido, simple, ideal para notebooks y prototipos | Aprendizaje, prototipado, datasets medianos |
| **Chroma** | Base de datos vectorial open-source. Persistencia en disco | Proyectos que necesitan persistencia sin infraestructura extra |
| **Pinecone** | Vector store cloud. Escalable, gestionado | Producción con millones de vectores |
| **Weaviate** | Base de datos vectorial con schema. Open-source | Producción con datos estructurados + vectores |

> **Todos comparten la misma interfaz**: `.from_texts()`, `.similarity_search()`, `.as_retriever()`.

## BLOQUE 2 — Setup

In [ ]:
import os, getpass
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()

DOCS = [
    "La politica de vacaciones es de 15 dias por ano.",
    "El seguro medico esta incluido desde el primer dia de trabajo.",
    "El horario es de 9 a 18 con 1 hora de almuerzo.",
    "El trabajo remoto esta permitido 3 dias por semana.",
    "Los bonos anuales se pagan en diciembre segun desempeno.",
    "Para solicitar vacaciones hay que completar el formulario HR-01.",
]

print(f"Documentos a indexar: {len(DOCS)}")

## BLOQUE 3 — TODO 1: Crear el índice FAISS

`FAISS.from_texts()` hace 3 cosas en un solo paso:

1. **Embede** cada texto con el modelo de embeddings
2. **Indexa** los vectores en una estructura de búsqueda rápida
3. **Guarda** el texto original junto con cada vector

```python
vectorstore = FAISS.from_texts(textos, embeddings)
#              ^- metodo de clase   ^- raw texts  ^- embedding model
```

También existe `FAISS.from_documents()` si ya tenés objetos `Document` en vez de strings.

In [ ]:
# TODO 1
vectorstore = FAISS.from_texts(DOCS, embeddings)
print(f"Tipo: {type(vectorstore).__name__}")
print(f"Cantidad de vectores indexados: {vectorstore.index.ntotal}")
print()
print("FAISS.from_texts() embebe cada texto y construye el indice.")

## BLOQUE 4 — TODO 2: Búsqueda por similitud

`similarity_search(query, k)` busca los `k` documentos más cercanos al vector de la consulta.

Internamente:
1. Embede la query con `embeddings.embed_query(query)`
2. Busca en el índice FAISS los `k` vectores más cercanos (usando similitud del coseno o distancia L2)
3. Devuelve los `k` objetos `Document` con el texto original

```text
Pregunta: "vacaciones"
  |
  v (embed_query)
  Vector de la pregunta
  |
  v (FAISS index search)
  [Doc: "La politica de vacaciones...",  <- mas cercano
   Doc: "Para solicitar vacaciones..."]  <- segundo mas cercano
```

In [ ]:
# TODO 2
docs = vectorstore.similarity_search("vacaciones", k=2)
print(f"Resultados para 'vacaciones' (k=2):")
for i, doc in enumerate(docs):
    print(f"  {i+1}. {doc.page_content}")
    print(f"     Score/distancia: (FAISS no expone score por defecto)")
print()
print("FAISS devuelve los textos cuyos vectores son mas cercanos semanticamente.")

## BLOQUE 5 — TODO 3: Probar distintas queries y valores de k

El parámetro `k` controla cuántos documentos recuperar:

| k | Ventaja | Desventaja |
|---|---|---|
| **k=1** | Máxima precisión, mínimo costo de tokens | Puede perder contexto relevante |
| **k=3** | Contexto más completo | Más tokens = mayor costo y posible ruido |
| **k=5+** | Cubre múltiples aspectos | Token count alto, el LLM puede distraerse |

En producción el valor típico es **k=3 a k=5**.

In [ ]:
# TODO 3
queries = ["dias de vacaciones", "trabajo desde casa", "pago de bonos"]
for query in queries:
    docs = vectorstore.similarity_search(query, k=1)
    print(f"Query: '{query}'")
    print(f"  Resultado: {docs[0].page_content}")
    print()

print("Cada query recupera el doc semanticamente mas cercano.")

## BLOQUE 6 — Persistencia: guardar y cargar el índice

FAISS permite guardar el índice en disco para no tener que re-indexar cada vez.

In [ ]:
import tempfile

# Guardar a disco
with tempfile.TemporaryDirectory() as tmp:
    vectorstore.save_local(tmp)
    print(f"Indice guardado en: {tmp}")

    # Cargar desde disco
    loaded = FAISS.load_local(tmp, embeddings, allow_dangerous_deserialization=True)
    print(f"Indice cargado: {type(loaded).__name__}")
    print(f"Documentos en indice cargado: {loaded.index.ntotal}")
    
    docs_cargados = loaded.similarity_search("vacaciones", k=1)
    print(f"Busqueda post-carga: {docs_cargados[0].page_content}")

print()
print("save_local() / load_local() permiten persistir el indice en disco.")

## BLOQUE 7 — FAISS vs búsqueda ingenua

| Aspecto | Búsqueda ingenua (sin vector store) | FAISS |
|---|---|---|
| **Mecanismo** | Comparar pregunta contra cada documento con un loop | Índice optimizado para búsqueda de vecinos cercanos |
| **Complejidad** | O(n) — recorre todos los documentos | O(log n) o mejor — usa índices particionados |
| **Escala** | 10-100 documentos | Millones de documentos |
| **Precisión** | Exhaustiva (compara todos) | Aproximada (puede perder algunos) |
| **Código** | Loop for manual | `vectorstore.similarity_search(query, k)` |

### Errores comunes

| Error | Consecuencia | Solución |
|---|---|---|
| `k` muy alto | Token count excesivo, el LLM recibe ruido | Empezá con k=3 y ajustá según calidad |
| `k` muy bajo | Respuesta incompleta | Probá k=1 vs k=3 y compará resultados |
| No persistir el índice | Re-indexar cada vez que ejecutás | Usá `save_local()` después de crear el índice |
| Usar embeddings distintos en indexación y consulta | Resultados sin sentido | Usá el MISMO objeto `embeddings` |

## BLOQUE 8 — Checks automáticos

In [ ]:
def run_checks():
    docs = vectorstore.similarity_search("vacaciones", k=2)
    assert len(docs) == 2
    contenidos = [d.page_content for d in docs]
    assert any("vacaciones" in c.lower() or "15 dias" in c.lower() for c in contenidos)
    docs_remoto = vectorstore.similarity_search("trabajo desde casa", k=1)
    assert "remoto" in docs_remoto[0].page_content.lower()
    print("M3L2 E08 Resolution checks passed")

run_checks()

## BLOQUE 9 — Comparación final

| Concepto | ¿Qué es? | ¿Para qué sirve? |
|---|---|---|
| **FAISS** | Índice de vectores de Facebook | Búsqueda eficiente de textos similares |
| **`.from_texts()`** | Método de clase que crea el índice | Indexar documentos + embeddings en un paso |
| **`.similarity_search()`** | Busca los k vectores más cercanos | Recuperar documentos relevantes para una query |
| **`k`** | Número de resultados a devolver | Controlar cantidad de contexto |
| **`save_local()`** | Persiste el índice en disco | No re-indexar cada vez |
| **`load_local()`** | Carga un índice desde disco | Reutilizar en otra sesión |

### Próximos pasos

- **E09 (Retriever)**: envolver FAISS en la interfaz estándar `as_retriever()`
- **E10 / E11 (RAG)**: pipeline completo con retriever + prompt + llm + parser